In [0]:
class Silver_pit_stops():
    main_path="/Volumes/formula1_race/default/formula1/"
    bronze_path = "formula1_race_project/bronze"
    silver_path = "formula1_race_project/silver"

    def __init__(self,folder):
         self.folder_name=folder
        
    def read_input(self):
        from pyspark.sql.functions import col,max,count
        if spark.catalog.tableExists("formula1_race.silver.pit_stops"):
            max_ingestion_date=spark.read.table('formula1_race.silver.pit_stops').agg(max(col('pit_stops_ingestion_date')).alias('max_ingestion_date')).collect()[0]['max_ingestion_date']
            if max_ingestion_date is None:
                max_ingestion_date='1900-01-01 00:00:00'
            print("if_max_ingestion_date:",max_ingestion_date)
            print(f"if_max_ingestion_date:",type(max_ingestion_date))
            read_df=spark.read.table('formula1_race.bronze.pit_stops').filter(col('PitStopsIngestionDate')>max_ingestion_date)
    
            print("reading pit_stops read_df")
            display(read_df.select(count("*")))
        else:
            max_ingestion_date='1900-01-01 00:00:00'
            print("else_max_ingestion_date:",max_ingestion_date)
            read_df=spark.read.table('formula1_race.bronze.pit_stops').filter(col('PitStopsIngestionDate')>max_ingestion_date)
        return read_df
    
    def column_name_formating(self,read_df):
        colrename_df=read_df
        import re
        for c in colrename_df.columns:
            result = re.sub(r'([a-z])([A-Z])',r'\1,\2',c).lower().split(',')
            new_column="_".join(result)
            colrename_df=colrename_df.withColumnRenamed(c,new_column)
        return colrename_df
    
    def apply_transformations(self,colrename_df):
        from pyspark.sql.functions import round,col
        apply_tran_df= (colrename_df.withColumn('duration_sec',round(col('milliseconds')/1000,4))
                        .withColumn('duration_minutes',round(col('milliseconds')/60000,4))
                        .select(col('driver_id'),col('race_id'),col('stop'),col('lap'),col('time'),col('duration_sec'),col('duration_minutes'),col('milliseconds'),col('pit_stops_ingestion_date'),col('source'))
                        )
    
        return apply_tran_df
    
    def write_output(self,apply_tran_df):
        apply_tran_df.write.partitionBy("race_id").mode("append").saveAsTable("formula1_race.silver.pit_stops")
        display(spark.sql(f"select count(*) from formula1_race.silver.pit_stops"))
        print("Data write into sliver pit_stops table is Done")
     
    def process(self):
        print("Started silver-ingestion-pit_stops  in ran....")
        read_df=self.read_input()
        colrename_df=self.column_name_formating(read_df)
        apply_tran_df=self.apply_transformations(colrename_df)
        self.write_output(apply_tran_df)

In [0]:
Silver_pit_stops_instance = Silver_pit_stops("pit_stops")
Silver_pit_stops_instance.process()